<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Cell 1 — Setup
!pip -q install duckdb huggingface_hub pandas

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Get token from Colab Secret
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face Read token "
        "in Colab Secrets with the exact name: HF_TOKEN"
    )

print("HF_TOKEN loaded.")

# DuckDB connection
con = duckdb.connect()

# Hugging Face authentication
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB connected.")

HF_TOKEN loaded.
DuckDB connected.


In [ ]:
# Cell 2 — Find warehouse files
# Find available warehouse parquet files

files = con.sql("""
SELECT file
FROM glob(
    'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
)
""").df()

print("Files found:", len(files))
display(files.head(30))

Files found: 22


,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [ ]:
# Cell 3 — Find February and March
feb_matches = files[
    files["file"].astype(str).str.contains("2026-02", regex=False)
]

mar_matches = files[
    files["file"].astype(str).str.contains("2026-03", regex=False)
]

print("February files:")
display(feb_matches)

print("March files:")
display(mar_matches)

February files:


,file
15,hf://datasets/FlyRank/internship-warehouse/fac...


March files:


,file
16,hf://datasets/FlyRank/internship-warehouse/fac...


In [ ]:
# Cell 4 — Set feature/label files

if len(feb_matches) == 0:
    raise ValueError("February 2026 file not found.")

if len(mar_matches) == 0:
    raise ValueError("March 2026 file not found.")

FEB = feb_matches.iloc[0]["file"]
MAR = mar_matches.iloc[0]["file"]

print("FEB:", FEB)
print("MAR:", MAR)

FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet
MAR: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
# Cell 5 — Inspect columns
# **YE CELL RUN KARNE KE BAAD ISKA OUTPUT SHARE KARNA HAI**
columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{FEB}')
LIMIT 1
""").df()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

One row represents a specific content item's performance metrics for a given client on a specific date. The feature window is February 2026, and all features are derived from data available within this month. The outcome or label will be measured in the subsequent month, March 2026, ensuring no overlap and preventing data leakage

In [ ]:
# Verification Query 1 — Grain
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS distinct_grain_rows
FROM read_parquet('{FEB}')
""").df()

display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,7355108,7355108


In [ ]:
# Verification Query 2 — Row count + date span
count_span_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM read_parquet('{FEB}')
""").df()

display(count_span_check)

,row_count,min_date,max_date,distinct_dates
0,7355108,2026-02-01,2026-02-28,28


In [ ]:
# Verification Query 3 — Availability
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet('{FEB}')
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,7355108,2621783


## 2. Fields: feature / label / context / excluded

**Feature fields:**
I will use fields that are available by the end of February 2026 and reflect content performance during that month. These include metrics like gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, and ga4_sessions. These fields provide observable data about how content performed in the feature window.

**Label:**
The label will represent a key performance indicator for the content in the following month, March 2026. For this exercise, let's define the label as ga4_sessions in March 2026. This is a common metric to predict, as it indicates user engagement.

**Context:**
report_date, client_hash_id, and content_hash_id are essential context fields. They define the temporal and entity scope of each row and are used for grouping and joining data.

**Excluded:**
I exclude fields that directly reveal the March outcome (e.g., ga4_sessions from March) or are inherently created after the decision moment. For example, I exclude any fields that aggregate data beyond February 2026. I also exclude client_has_gsc and client_has_ga4 as they are static client-level flags and not specific to content performance at the row level. gsc_data_available and ga4_data_available are used for filtering, not as direct features.

## 3. Verify it with queries (grain, counts, missing values, windows)

The queries for grain , counts and availability are already provided above in section 1

## 4. Data limits

This data slice has limitations. Firstly, it does not capture the full causal factors behind content performance; it provides correlational insights. Secondly, the availability of GSC and GA4 data might vary across clients and dates, meaning not all rows have complete data from both sources. Thirdly, the outcome (March 2026 performance) is inherently unknown at the time of decision-making in February 2026, which is a fundamental aspect of predictive modeling but limits what can be known beforehand. These limitations mean the features should be used for decision-support rather than definitive causal analysis.

## Self-check


*   Every section above is filled — markdown thinking AND the code that backs it.
*   The notebook runs top to bottom with no errors (Runtime → Run all).
*   No client names, URLs, or private queries anywhere.
*   My claims use careful words: observed, measured, directional, decision-support.
*   Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.